# 8.6 遇到模糊指令怎麼辦？


使用者說「幫我安排會議」，卻沒有告訴你哪一天。直接說「已安排明天」可能做錯；反過來，每次都問十個問題也會拖慢本來很簡單的事。本節要找的是會改變下一步的必要資訊，缺少它才澄清。這與[8.5的遵循明確限制](https://birdhackor.github.io/tiny-perceptron-vlm/8.5.html)接續：限制明確就照做，關鍵條件模糊就先把問題縮小。

先備是[暖身的Python字典與迴圈](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。這裡不用真行事曆，也不發出會議邀請，只設計一個小世界：任務僅為「確認日期」。在這個範圍中，日期是必要欄位，沒日期不能完成；有日期就不該重複追問。若任務變成真的訂會議，時間和參與者也可能變成必要欄位，必須另外定義。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
requests = [
    {"task": "確認日期", "date": None},
    {"task": "確認日期", "date": "2026-10-03"},
]
for request in requests:
    if request["date"] is None:
        target = "你希望哪一天？"
    else:
        target = "已確認日期：" + request["date"]
    print(request, "→", target)

輸入兩個字典的任務相同，只有日期不同。`None`是Python用來表示這裡沒有值的符號，不是日期字串。關鍵判斷`is None`檢查是否缺日期。`if`在判斷成立時執行它下面縮排的目標句；`else`在判斷不成立時才執行另一句。因此缺日期時設定一個只問日期的目標回答，否則把日期接在確認句後，兩種分支最後都會到`print`。輸出第一筆對應「你希望哪一天？」，第二筆對應「已確認日期：2026-10-03」。注意第二筆沒有再問使用者是哪一天。

這段是一個人寫好的標註規則：我們知道小世界的必要欄位，所以能產生理想回答，後續可把它做成對話訓練資料。它不是模型自主辨識模糊指令的展示。若把規則輸出當成模型能力，就省略了模型是否真的從新措辭讀出日期、是否能分清必要與可預設條件的重要測試。

不同任務的必要資訊不同。「把2+2算出來」不需要先問喜歡什麼語氣；「選適合你的交通方案」可能需要目的地。可以合理預設的細節應說明預設，例如使用普通書面語；會讓結果不可用的缺項則應問。訓練資料要同時包含缺資訊而澄清、資訊足夠而直接完成的正例，否則模型可能把「勤問問題」誤學成永遠不動手。

這種成對日期題也加入了[8.4的同一份模型](https://birdhackor.github.io/tiny-perceptron-vlm/8.4.html)。24個日期各有「日期已給」與「日期缺少」兩題，先把同日期的兩題視為一組，再把不同日期組分給三種用途：19個日期、38題用來更新模型；2個日期、4題用來觀察學習中的表現，叫驗證資料；另外3個日期、6題留到最後檢查，不拿來更新模型。兩題一起分配，避免模型在學習時見過某日期，最後卻又拿同日期的另一題當新題。缺日期題在資料表中仍保留所屬日期的分組標記，但輸入的`date`欄位用`?`代替實際日期，另帶題目編號`id`，不是把分組日期直接告訴模型。

最後檢查的三個日期都未用於訓練，因此是在核對模型能否處理新日期。缺日期的三題都生成`請提供日期。`；但日期已給的三題全確認錯了。例如`task=date;date=2026-10-01;confirm`得到`已確認2026-10-10。`。所以3／6的總分包含兩種很不同的結果：模型學到了固定澄清句，還不能可靠保留新日期。這不是已能安排會議，也沒有測試任意自然語句的缺資訊判斷；它讓我們知道下一步要補哪種能力。

練習讓小世界改為「確認會議日期與時間」：替兩筆字典加`time`，第一筆用`None`、第二筆用`"10:00"`。先在紙上寫出四種日期/時間有無的情況要問什麼，再修改條件判斷。核對時，兩項都齊就確認，缺哪項才問哪項；不要把缺時間的任務仍當成完成安排。
